In [ ]:
!pip install -U langchain langchain-community langchain-core langchain-huggingface langchain-groq langchain-chroma chromadb sentence-transformers

In [ ]:
!pip install -qU langchain-classic

In [ ]:
from google.colab import drive
import os

# Montar Google Drive para persistencia de la base de datos ChromaDB
drive.mount('/content/drive')

# Definir el directorio de trabajo en Drive
WORK_DIR = '/content/drive/MyDrive/Proyecto_RAG_Coyuntura'
DB_DIR = os.path.join(WORK_DIR, 'chroma_db')
DATA_DIR = os.path.join(WORK_DIR, 'datos')

# Crear directorios si no existen
os.makedirs(DB_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Directorio de base de datos configurado en: {DB_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de base de datos configurado en: /content/drive/MyDrive/Proyecto_RAG_Coyuntura/chroma_db


In [ ]:
from google.colab import userdata
import os

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ Clave de Groq cargada correctamente.")
except Exception as e:
    print("⚠️ Error: No olvides configurar 'GROQ_API_KEY' en la sección de Secrets.")

✅ Clave de Groq cargada correctamente.


In [ ]:
%%writefile motor_rag.py
import pandas as pd
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
         "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]


class PipelineCoyuntura:
    def __init__(self, directorio_db: str):
        self.directorio_db = directorio_db
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        )
        self.llm = ChatGroq(
            model_name="llama-3.1-8b-instant",
            temperature=0.2,
            api_key=os.environ.get("GROQ_API_KEY")
        )

    def indexar_corpus(self, ruta_csv: str, reiniciar: bool = True):
        # Evita duplicados: borra la base vectorial existente antes de reindexar
        if reiniciar and os.path.exists(self.directorio_db):
            import shutil
            shutil.rmtree(self.directorio_db)

        df = pd.read_csv(ruta_csv)
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

        chunks = []
        chunk_ids = []
        for idx, fila in df.iterrows():
            fecha = str(fila['Fecha'])
            mes = MESES[pd.to_datetime(fila['Fecha']).month - 1]
            titulo = str(fila['Título'])
            url = str(fila['Url'])

            # Se divide primero el cuerpo de la noticia...
            fragmentos_texto = text_splitter.split_text(str(fila['Cuerpo']))

            # ...y se antepone la fecha a CADA fragmento, no solo al primero
            for i, frag in enumerate(fragmentos_texto):
                chunks.append(
                    Document(
                        page_content=f"Fecha: {fecha}. Noticia: {frag}",
                        metadata={"fecha": fecha, "mes": mes, "titulo": titulo, "url": url}
                    )
                )
                # id determinístico: reindexar el mismo artículo sobreescribe en vez de duplicar
                chunk_ids.append(f"{idx}-{i}")

        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.directorio_db,
            ids=chunk_ids
        )
        return len(chunks)

    def cargar_motor(self):
        self.vectorstore = Chroma(
            persist_directory=self.directorio_db,
            embedding_function=self.embeddings
        )
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})

        system_prompt = (
            "Eres un analista de datos experto en coyuntura política boliviana. "
            "Tu tarea principal es realizar análisis cronológicos precisos. "
            "REGLA DE ORO: Cada fragmento de contexto tiene una fecha al inicio. "
            "Antes de responder, filtra los fragmentos que NO correspondan al mes o periodo solicitado por el usuario. "
            "Si un evento ocurrió en un mes distinto, descártalo para el análisis solicitado. "
            "Identifica los actores políticos mencionados únicamente en el periodo relevante. "
            "Cita siempre las fuentes con su fecha exacta.\n\n"
            "Contexto recuperado:\n{context}"
        )

        prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])
        question_answer_chain = create_stuff_documents_chain(self.llm, prompt)
        self.chain = create_retrieval_chain(self.retriever, question_answer_chain)

    def consultar(self, pregunta: str, mes_filtro: str = None) -> dict:
        # Reconfiguración dinámica de los parámetros de búsqueda según el filtro
        search_kwargs = {"k": 5}
        if mes_filtro:
            search_kwargs["filter"] = {"mes": mes_filtro.lower()}

        self.retriever.search_kwargs = search_kwargs

        return self.chain.invoke({"input": pregunta})

Overwriting motor_rag.py


In [ ]:
import sys
import os

# Forzamos la recarga del módulo si hubo cambios
if 'motor_rag' in sys.modules:
    del sys.modules['motor_rag']

from motor_rag import PipelineCoyuntura

RUTA_CSV = '/content/drive/MyDrive/Proyecto_RAG_Coyuntura/datos/noticias.csv'
DB_DIR = '/content/drive/MyDrive/Proyecto_RAG_Coyuntura/chroma_db'

if os.path.exists(RUTA_CSV):
    print("Iniciando indexación...")
    pipeline = PipelineCoyuntura(directorio_db=DB_DIR)
    total_chunks = pipeline.indexar_corpus(RUTA_CSV)
    print(f"✅ ¡Éxito! Se indexaron {total_chunks} fragmentos.")
else:
    print(f"⚠️ No se encontró el archivo en {RUTA_CSV}.")
    print("Verifica que el archivo exista en tu Drive en esa ruta exacta.")

Iniciando indexación...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ ¡Éxito! Se indexaron 9422 fragmentos.


In [ ]:
# en notebook 2, al final
!cp motor_rag.py /content/drive/MyDrive/Proyecto_RAG_Coyuntura/